# 206. Reverse Linked List

**Easy**

Given the `head` of a singly linked list, reverse the list, and return *the new
head*.

---

**Example 1:**

```
Input:  head = 1 -> 2 -> 3 -> 4 -> 5 -> None
Output: 5 -> 4 -> 3 -> 2 -> 1 -> None
```

**Example 2:**

```
Input:  head = 1 -> 2 -> None
Output: 2 -> 1 -> None
```

**Example 3:**

```
Input:  head = None
Output: None
```

---

**Constraints:**

- The number of nodes in the list is in the range `[0, 5000]`.
- `-5000 <= Node.val <= 5000`

**Follow-up:** you can do it iteratively **or** recursively. Do both.


### You have already written half of this

Remember your `LinkedList.addFirst` from the zigzag notebook? Every time you
added a node it went to the **front**. So if you walk the old list `1 -> 2 -> 3`
and `addFirst` each value into a new list, you get `3 -> 2 -> 1`. Reversal falls
out for free. That is one honest solution - `O(n)` time, `O(n)` extra space for
the new list.

But the classic answer reverses **in place** with `O(1)` extra space - no new
list, just re-pointing the arrows of the list you already have. Three pointers:

```
prev = None
curr = head
while curr:
    (1) remember where the rest of the list is       <- before you overwrite anything
    (2) flip curr's arrow to point back at prev
    (3) slide prev and curr forward one step
return prev
```

The one question that matters: in step (2) you overwrite `curr.next`. If you did
**not** save it first in step (1), what happens to the rest of the list `3 -> 4 -> 5`?
Draw three boxes and three arrows and walk one iteration by hand - that picture
*is* the algorithm.

- Why does the loop return `prev` and not `curr` at the end? (Where is `curr`
  sitting when the loop stops?)
- What does this return for the empty list, with no special-casing? Check that
  `prev` starts in the right place.

**Then the recursive version:** reverse everything after `head`, then make the
node *after* head point back at head. It is four lines and slightly mind-bending
- worth the stretch.

### The node definition

LeetCode's standard singly-linked-list node. Note it is **not** your `TreeNode` -
one `next`, no `left`/`right`. Run this first.

In [30]:
# Definition for singly-linked list.
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next
    def printList(self):
        current = self
        while current:
            print(current.val, end=" ")
            current = current.next
        print("None")

class Stack:
    def __init__(self ):
        self.head = None
    def push(self, i:ListNode):
        i.next = self.head
        self.head = i
    def printStack(self):
        current = self.head
        while current:
            print(current.val, end=" ")
            current = current.next
        print("--> None")

class Solution:
    def reverseList(self, head: ListNode) -> ListNode:
        stack = Stack()
        temp:ListNode = head
        while temp:
            current = temp.next
            stack.push(temp)
            temp = current
        return stack.head
    def reverseList2(self, head: ListNode) -> ListNode:
        if head == None or head.next is None: return head
        new_head = self.reverseList(head.next)
        head.next.next = head
        head.next = None
        return new_head


head = ListNode(10)
head.next = ListNode(20)
head.next.next = ListNode(30)
head.next.next.next = ListNode(40)
head.next.next.next.next = ListNode(50)
head.next.next.next.next.next = ListNode(60)
solution = Solution()
head.printList()
solution.reverseList2(head).printList()

10 20 30 40 50 60 None
60 50 40 30 20 10 None


### Two helpers - build a list, and read one back

`build(...)` turns a Python list into a chain of `ListNode`s; `to_list(...)`
walks a chain back into a Python list so the tests are readable. Run this cell;
don't edit it.

In [ ]:
def build(values):
    """Python list -> linked list, return the head."""
    head = None
    for v in reversed(values):     # build back-to-front so order is preserved
        head = ListNode(v, head)
    return head


def to_list(head):
    """Linked list -> Python list, so we can print it."""
    out = []
    while head:
        out.append(head.val)
        head = head.next
    return out


In [ ]:
# tests
sol = Solution()

print(to_list(sol.reverseList(build([1, 2, 3, 4, 5]))))   # [5, 4, 3, 2, 1]
print(to_list(sol.reverseList(build([1, 2]))))            # [2, 1]
print(to_list(sol.reverseList(build([]))))                # []
print(to_list(sol.reverseList(build([7]))))               # [7]
print(to_list(sol.reverseList(build([1, 1, 2, 2]))))      # [2, 2, 1, 1]

# a longer one - make sure nothing is dropped or duplicated
n = 1000
big = list(range(n))
out = to_list(sol.reverseList(build(big)))
print(out == big[::-1], "len", len(out))                  # True len 1000


### The trap to watch for

The number-one bug here is losing the tail. The instant you write
`curr.next = prev` **before** saving `curr.next`, the rest of the list vanishes -
you are standing on a branch and sawing it off behind you. If your `[1,2,3,4,5]`
comes back as `[1]` or hangs forever, that is the bug. Save `next` first, every
time.